# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available RecordSets by their @id and name
print("Available RecordSets:")
for record_set in metadata.record_sets:
    print(f"- {record_set['@id']}: {record_set['name']}")

# For this dataset, let's list fields for each record set
for record_set in metadata.record_sets:
    print(f"\nFields in RecordSet {record_set['@id']} ({record_set['name']}):")
    for field in record_set['fields']:
        print(f"  - Field @id: {field['@id']}, name: {field['name']}, dataType: {field.get('dataType','')} column: {field['column'] if 'column' in field else 'N/A'}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract all dataframes from all the record sets
record_sets = [record_set['@id'] for record_set in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if len(df) > 0:
        print(f"Columns in {record_set_id}: {list(df.columns)}\nExample records:")
        display(df.head(2))
    else:
        print(f"No records found in {record_set_id}.")

# For demonstration, we'll focus analysis on the first RecordSet
main_record_set_id = record_sets[0] if len(record_sets) > 0 else None
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Using RecordSet: {main_record_set_id} for further analysis.")
else:
    raise ValueError("No record sets found in the schema.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Infer numeric fields to use for filtering and normalization
import numpy as np

# Find all numeric columns
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_fields:
    # Try to coerce potential numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except:
            pass
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Numeric fields detected: {numeric_fields}")
if len(numeric_fields) == 0:
    raise ValueError("No numeric fields found for EDA.")

# Pick the first numeric field for example analysis
numeric_field = numeric_fields[0]

threshold = df[numeric_field].median() if df[numeric_field].dtype.kind in 'fi' else 10  # Use median
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by another field if a categorical field is present
categorical_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = None

# Pick the first categorical field with < 20 unique values (likely a group)
for col in categorical_fields:
    if df[col].nunique() < 20:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field}, showing mean of {numeric_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the main numeric field (before filter)
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# Boxplot (after filter and normalization)
if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=f"{numeric_field}_normalized", data=filtered_df)
    plt.title(f"Boxplot of normalized {numeric_field} by {group_field}")
    plt.show()

# If additional numeric fields exist, visualize correlations
if len(numeric_fields) > 1:
    plt.figure(figsize=(8, 6))
    sns.heatmap(df[numeric_fields].corr(), annot=True, cmap='coolwarm')
    plt.title('Correlation matrix of numeric fields')
    plt.show()

## 6. Conclusion
This notebook demonstrated accessing the FAIR² Croissant dataset of mass spectrometry protein analysis, loading record sets and fields by their `@id`, and performing basic EDA. We filtered records by a numeric field, normalized data, grouped by a categorical attribute, and visualized distributions to quickly check the dataset structure.

**Next steps:** You can extend this analysis to more specific questions, further feature engineering, or modeling tasks, referencing dataset attributes directly by their stable `@id` from the Croissant schema.